# Energy Spectrum of Vela

## Imports

In [32]:
# Imports
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

## Arguments

In [33]:
dR = 1.5 # degrees, side length of square region of interest
lon_Vela = 263.55
lat_Vela = -2.79

name = "Vela"

## Global variables

In [34]:
dirDir = "/home/users/l/lebrun/"
resultPath = dirDir + "results/"
dataPath = dirDir + "data/"
figPath = dirDir + "figures/Spectrums/"
filename = dataPath + "DmlNtup_Data_8years_FullPhotonData_AllSky.root"

## Load Data

In [35]:
tree = None
f = ROOT.TFile(filename, "READ") 

tree = f.Get("DmlNtup")

df = ROOT.RDataFrame(tree)

In [36]:
%%cpp 

double rad2deg(double x) {
    return x / (2*M_PI) * 360;
}

In [37]:
df = df.Define("longitude_deg", "rad2deg(celestialcoordBest_2)").Define("latitude_deg", "rad2deg(celestialcoordBest_3)")

## Filter on Object ROI

In [38]:
lon_min = lon_Vela - dR
lon_max = lon_Vela + dR
lat_min = lat_Vela - dR
lat_max = lat_Vela + dR

dfROI = df.Filter(f"longitude_deg >= {lon_min}").Filter(f"longitude_deg <= {lon_max}").Filter(f"latitude_deg >= {lat_min}").Filter(f"latitude_deg <= {lat_max}")
print(f"Number of entries : {dfROI.Count().GetValue()}") # Number of entries

Number of entries : 5916


## Compute Energy Spectrum

### Load acceptance

In [39]:
# Load acceptance       
acceptance = np.loadtxt(dataPath+"acceptance_vs_energy_cutBDT_v25.txt") # 100 values

ene = np.logspace(1, 5, 101) / 10 

from scipy import interpolate

def create_acceptance_function(energy_edges, acceptance_values):
    """
    Creates an interpolation function that returns acceptance values
    for given energy inputs.
    
    Parameters:
    energy_values (array): Array of energy values in log scale
    acceptance_values (array): Array of corresponding acceptance values
    
    Returns:
    function: Interpolation function that takes energy as input and returns acceptance
    """
    # Calculate bin centers (geometric mean for log-spaced bins)
    energy_centers = np.sqrt(energy_edges[:-1] * energy_edges[1:])
    # Create interpolation function
    # We use 'linear' interpolation which works well for log-scaled data
    interp_func = interpolate.interp1d(
        energy_centers,
        acceptance_values,
        kind='linear',
        bounds_error=False,  # Don't raise error for out-of-bounds values
        fill_value=(acceptance_values[0], acceptance_values[-1])  # Extrapolate using edge values
    )
    
    return interp_func

# Create the function
acceptance_function = create_acceptance_function(ene, acceptance)  # Note: ene has 101 values but acceptance has 100


### Compute acceptance at each point

In [40]:
# Filter from lbound to 1e5

l_bound = 2 # GeV

dfLog = dfROI.Define("log10_E", "TMath::Log10(tt_bgoTotalE_GeV)")
dfAcc = dfLog.Filter("log10_E < 5").Filter(f"tt_bgoTotalE_GeV >= {l_bound}")

dfNpy = dfAcc.AsNumpy(["tt_bgoTotalE_GeV"])
energies = dfNpy["tt_bgoTotalE_GeV"]
acceptances = acceptance_function(energies)

dfNpy["acceptance"] = acceptances
dfNpy["acc_weight"] = 1/acceptances
dfNpy["energyCorr"] = energies/acceptances

dfAcc = ROOT.RDF.FromNumpy(dfNpy)
dfAcc = dfAcc.Define("log10E_Corr", "TMath::Log10(energyCorr)")


### Create histogram of acceptance-corrected energy

In [41]:
%%cpp
double powerLaw(double *x, double a, double b){
    return b*pow(x[0], a);
}

In [42]:
%%cpp
double fitFunction(double *x, double *par){
    return powerLaw(x, par[0], par[1]);
}

In [43]:

hAcc = dfAcc.Histo1D(("Ecorr", f"Energy [E > {l_bound} GeV];Energy [GeV];Counts/Acceptance [cm^{{-2}}sr^{{-1}}]", 800, l_bound, 110),
                      "tt_bgoTotalE_GeV", "acc_weight")
fitFuncE = ROOT.TF1("fitFunction", ROOT.fitFunction, xmin=l_bound, xmax=100, npar=2)
fitFuncE.SetParameter(0, -2.5) # a
fitFuncE.SetParameter(1, 4) # b 

res = hAcc.Fit(fitFuncE, "R")

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      143.657
NDf                       =          152
Edm                       =  1.87518e-06
NCalls                    =          217
p0                        =     -3.29692   +/-   0.0286163   
p1                        =       103114   +/-   4277.54     


### Plot it

In [44]:
ROOT.gStyle.SetOptStat(0)

c = ROOT.TCanvas()
hAcc.Draw()

alpha = fitFuncE.GetParameter(0)
beta = np.log10(fitFuncE.GetParameter(1))

leg = ROOT.TLegend(0.5, 0.7, 0.85, 0.85)

leg.AddEntry(hAcc.GetPtr(), "Total Reconstructed Energy", "lep")
leg.AddEntry(fitFuncE, f"log_{{10}}(y) = {alpha:.2f}log_{{10}}(x) + {beta:.2f}")

leg.SetTextSize(0.03)
leg.Draw("same")
c.SetLogx()
c.SetLogy()
c.SetGrid()
c.SaveAs(figPath+f"{name}_Spectrum.pdf")
c.Draw()

Info in <TCanvas::Print>: pdf file /home/users/l/lebrun/figures/Spectrums/Vela_Spectrum.pdf has been created
